# Runtime and memory: Weighted CC vs Split CC

Reviewer: *"The paper reports neither wall-clock runtimes, scaling behavior, and memory requirements nor comparisons with the runtime of split conformal clustering."*

Setup of `Results/GMM2D5v1kNN.py`: $K=5$, means $4(\cos\theta_k,\sin\theta_k)$, covariance $\sigma^2 I$ with $\sigma^2=2.6$ (as in `GMM2D5v1kNN_asymp.py`), equal weights, $n_{tr}=n_{cal}=n/2$, kNN classifier with $k=\lfloor\sqrt{n_{tr}}\rfloor$, `'GMMS'` with `max_iter=200, n_init=5`, $\alpha = 0.1$.

**Task compared in Tables 1 and 2.** Given the trained clustering and classifier (identical code for both methods, reported once as *training, shared*), produce prediction sets for the same $m = 500$ test points.
* Split CC: one clustering fit on $\mathcal{D}_{cal}$, then the sets for the $m$ points (one batch), using the same trained classifier (`'pretrain'`).
* Weighted CC: for each of the $m$ points, a clustering fit on $\mathcal{D}_{cal}\cup\{x\}$ and the set.

Single process, one thread (`threadpool_limits(1)`). Only the existing calls are timed; `utils.py` is not modified. Nothing is saved to disk.

In [1]:
import sys, os, time, copy, platform, subprocess, tracemalloc
sys.path.append('../')

import numpy as np
import pandas as pd
import scipy, sklearn
from threadpoolctl import threadpool_limits

from weighted_conformal_clustering.utils import WeightedConformalClustering, ConformalClustering, sample_gmm

def _sysctl(key):
    try:
        return subprocess.run(['sysctl', '-n', key], capture_output=True, text=True).stdout.strip()
    except Exception:
        return ''

cpu = _sysctl('machdep.cpu.brand_string') or platform.processor()
mem = _sysctl('hw.memsize')
mem = f'{int(mem) / 2**30:.0f} GB' if mem.isdigit() else 'n/a'

_tp = threadpool_limits(limits=1)            # single-threaded BLAS/OpenMP from here on

### Settings

In [2]:
QUICK = False                                # True: a quick check (under a minute) that every cell runs

K = 5
_ANG = np.pi * np.linspace(0, 2, K, endpoint=False)
means = 4 * np.vstack([np.cos(_ANG), np.sin(_ANG)]).T
VAR = 2.6
covariances = np.array([np.eye(2) for _ in range(K)]) * VAR
weights = np.ones(K) / K
ALPHA = 0.1
gmm_kwargs = dict(n_components=K, max_iter=200, n_init=5)

N_RANGE = [200, 300, 500, 800, 1200, 1800, 2700, 4000]   # grid of GMM2D5v1kNN_asymp.py
TASK_M = 500                                 # test points in the task (the same points for both methods)
R_TASK = 5                                   # data seeds for Table 1; Table 2 uses seed 0

if QUICK:
    N_RANGE, TASK_M, R_TASK = [200, 500, 1200], 10, 2

### Helpers

In [3]:
def timed(f, *args, **kwargs):
    # (output, wall-clock seconds)
    t0 = time.perf_counter()
    out = f(*args, **kwargs)
    return out, time.perf_counter() - t0

def traced(f, *args, **kwargs):
    # (output, peak memory traced by tracemalloc during the call in MB); tracemalloc must be running
    tracemalloc.reset_peak()
    base = tracemalloc.get_traced_memory()[0]
    out = f(*args, **kwargs)
    return out, (tracemalloc.get_traced_memory()[1] - base) / 2**20

def gmm_data(n, m, seed):
    X, _ = sample_gmm(means, covariances, weights, n_samples=n + m, seed=seed)
    return X[:n // 2], X[n // 2:n], X[n:]     # training, calibration, test

def train_wcc(X_tr, X_cal):
    # as in GMM2D5v1kNN.py: retrain with a new random_state until the classifier sees all K clusters
    wcc = WeightedConformalClustering(X_tr, X_cal, alpha=ALPHA)
    wcc.set_classifier('kNN', n_neighbors=int(np.sqrt(len(X_tr))))
    for attempt in range(50):
        try:
            wcc.train('GMMS', **gmm_kwargs, random_state=attempt)
            return wcc
        except ValueError:
            continue
    raise RuntimeError('training failed')

def make_split(wcc, X_tr, X_cal):
    # Split CC with the same trained classifier (set-up, not part of the task)
    cc = ConformalClustering(X_tr, X_cal, alpha=ALPHA)
    cc.set_classifier('pretrain', clf=copy.deepcopy(wcc.clf))
    return cc

def split_task(cc, X_test):
    # Split CC task: calibration clustering fit, then the sets for all test points in one batch
    cc.fit('GMMS', **gmm_kwargs, random_state=0)
    return cc.conformal_set(X_test, method='narc')

def wcc_task(wcc, X_test):
    # Weighted CC task: one refit on D_cal ∪ {x} and one set for each test point
    return [wcc.conformal_set(X_test[[i]], 'GMMS', **gmm_kwargs, random_state=0) for i in range(len(X_test))]

def mean_sd(v, digits=3):
    v = np.asarray(v, float)
    return f'{v.mean():.{digits}g}' + (f' ({v.std(ddof=1):.2g})' if len(v) > 1 else '')

def print_latex(df):
    try:
        print(df.to_latex())
    except Exception as e:                   # DataFrame.to_latex needs jinja2
        print('to_latex unavailable:', e)

### Configuration reported with Tables 1 and 2

In [4]:
config = pd.DataFrame([
    ['machine', f'{cpu}, {os.cpu_count()} cores, {mem} RAM; {platform.platform()}'],
    ['software', f'Python {platform.python_version()}, numpy {np.__version__}, scipy {scipy.__version__}, '
                 f'scikit-learn {sklearn.__version__}'],
    ['execution', 'single process, one thread (threadpoolctl limit 1); wall-clock from time.perf_counter'],
    ['data', f'2D GMM, K = {K}, means 4(cos θ_k, sin θ_k) (θ_k equispaced), covariance σ² I with σ² = {VAR}, equal weights'],
    ['sample sizes', f'n in {N_RANGE}, n_tr = n_cal = n/2'],
    ['test points', f'm = {TASK_M}, the same points for both methods'],
    ['clustering', f"stochastic GMM ('GMMS'): GaussianMixture(n_components={K}, covariance_type='full', "
                   f"max_iter={gmm_kwargs['max_iter']}, n_init={gmm_kwargs['n_init']})"],
    ['classifier', 'k-nearest neighbours, k = floor(sqrt(n_tr)); Split CC uses the same trained classifier'],
    ['conformal', f'level 1 - α with α = {ALPHA}, non-randomised APS score'],
    ['repeats', f'Table 1: {R_TASK} data seeds, mean (sd); Table 2: seed 0'],
    ['memory', 'peak memory traced by tracemalloc during the task (Python and numpy allocations; lower bound, '
               'excludes the interpreter and library baseline common to both methods)'],
], columns=['item', 'value']).set_index('item')
with pd.option_context('display.max_colwidth', None):
    display(config)

,value
item,
machine,"Apple M2, 8 cores, 16 GB RAM; macOS-27.0-arm64-arm-64bit"
software,"Python 3.11.15, numpy 2.4.3, scipy 1.17.1, scikit-learn 1.8.0"
execution,"single process, one thread (threadpoolctl limit 1); wall-clock from time.perf_counter"
data,"2D GMM, K = 5, means 4(cos θ_k, sin θ_k) (θ_k equispaced), covariance σ² I with σ² = 2.6, equal weights"
sample sizes,"n in [200, 300, 500, 800, 1200, 1800, 2700, 4000], n_tr = n_cal = n/2"
test points,"m = 500, the same points for both methods"
clustering,"stochastic GMM ('GMMS'): GaussianMixture(n_components=5, covariance_type='full', max_iter=200, n_init=5)"
classifier,"k-nearest neighbours, k = floor(sqrt(n_tr)); Split CC uses the same trained classifier"
conformal,"level 1 - α with α = 0.1, non-randomised APS score"


### Table 1: time for the task across $n$

For each $n$ and data seed: training (shared), Split CC (calibration fit + sets for the $m$ test points), Weighted CC (sets for the same $m$ test points). Ratio = Weighted CC / Split CC within each seed. Mean (sd) over seeds.

In [5]:
t_sec = time.perf_counter()
rows = []
for n in N_RANGE:
    for seed in range(R_TASK):
        X_tr, X_cal, X_test = gmm_data(n, TASK_M, seed)
        wcc, t_train = timed(train_wcc, X_tr, X_cal)
        cc = make_split(wcc, X_tr, X_cal)
        _, t_split = timed(split_task, cc, X_test)
        _, t_wcc = timed(wcc_task, wcc, X_test)
        rows.append(dict(n=n, train=t_train, split=t_split, wcc=t_wcc))
    print(f'n = {n} done ({time.perf_counter() - t_sec:.0f}s)')
task_time = pd.DataFrame(rows)

table1 = pd.DataFrame([
    {'n': n, 'n_cal': n - n // 2,
     'training, shared (s)': mean_sd(g['train']),
     f'Split CC, {TASK_M} test points (s)': mean_sd(g['split']),
     f'Weighted CC, {TASK_M} test points (s)': mean_sd(g['wcc']),
     'ratio WCC / Split CC': mean_sd(g['wcc'] / g['split'])}
    for n, g in task_time.groupby('n')]).set_index('n')
print(f'Table 1. Time for prediction sets of the same {TASK_M} test points; mean (sd) over {R_TASK} data seeds; single core')
display(table1)
print_latex(table1)
print(f'section time {time.perf_counter() - t_sec:.0f}s')

n = 200 done (42s)
n = 300 done (81s)
n = 500 done (113s)
n = 800 done (146s)
n = 1200 done (189s)
n = 1800 done (242s)
n = 2700 done (315s)
n = 4000 done (425s)
Table 1. Time for prediction sets of the same 500 test points; mean (sd) over 5 data seeds; single core


,n_cal,"training, shared (s)","Split CC, 500 test points (s)","Weighted CC, 500 test points (s)",ratio WCC / Split CC
n,,,,,
200,100,0.02 (0.0097),0.018 (0.0061),8.37 (2.4),472 (36)
300,150,0.0142 (0.0025),0.0143 (0.0029),7.77 (1.1),549 (33)
500,250,0.0114 (0.0026),0.012 (0.0016),6.42 (1.4),534 (53)
800,400,0.0144 (0.0068),0.0144 (0.0004),6.56 (0.22),456 (15)
1200,600,0.0128 (0.0011),0.0177 (0.0016),8.61 (0.77),488 (24)
1800,900,0.0147 (0.0013),0.0219 (0.0017),10.4 (0.51),480 (50)
2700,1350,0.0196 (0.0015),0.0309 (0.002),14.5 (0.5),472 (24)
4000,2000,0.0259 (0.0019),0.0461 (0.0034),22 (0.52),478 (31)


to_latex unavailable: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
section time 425s


### Table 2: peak memory for the task across $n$

Same task as Table 1 (data seed 0). Split CC's peak is over its calibration fit and the batch of $m$ sets; Weighted CC's is over its $m$ refits (one test point at a time, so it does not grow with $m$).
Memory allocated through Python and numpy, traced by `tracemalloc` (a lower bound: memory allocated directly in C, and the baseline of the interpreter and libraries common to both methods, are not counted). Run separately from Table 1 because tracing slows the code down. With $P$ parallel workers each worker holds its own copy, so memory multiplies by about $P$.

In [6]:
t_sec = time.perf_counter()
memr = []
tracemalloc.start()
for n in N_RANGE:
    X_tr, X_cal, X_test = gmm_data(n, TASK_M, 0)
    wcc, m_train = traced(train_wcc, X_tr, X_cal)
    cc = make_split(wcc, X_tr, X_cal)
    _, m_split = traced(split_task, cc, X_test)
    _, m_wcc = traced(wcc_task, wcc, X_test)
    memr.append({'n': n, 'n_cal': n - n // 2, 'training, shared (MB)': m_train,
                 f'Split CC, {TASK_M} test points (MB)': m_split,
                 f'Weighted CC, {TASK_M} test points (MB)': m_wcc})
    print(f'n = {n} done ({time.perf_counter() - t_sec:.0f}s)')
tracemalloc.stop()

table2 = pd.DataFrame(memr).set_index('n').round(2)
print(f'Table 2. Peak traced memory for prediction sets of the same {TASK_M} test points (data seed 0); single core')
display(table2)
print_latex(table2)
print(f'section time {time.perf_counter() - t_sec:.0f}s')

n = 200 done (54s)
n = 300 done (87s)
n = 500 done (111s)
n = 800 done (134s)
n = 1200 done (161s)
n = 1800 done (192s)
n = 2700 done (231s)
n = 4000 done (280s)
Table 2. Peak traced memory for prediction sets of the same 500 test points (data seed 0); single core


,n_cal,"training, shared (MB)","Split CC, 500 test points (MB)","Weighted CC, 500 test points (MB)"
n,,,,
200,100,0.06,0.21,0.49
300,150,0.05,0.23,0.40
500,250,0.07,0.27,0.59
800,400,0.11,0.43,0.66
1200,600,0.16,0.67,1.01
1800,900,0.24,1.20,1.61
2700,1350,0.36,2.11,2.37
4000,2000,0.52,3.74,4.17


to_latex unavailable: Missing optional dependency 'Jinja2'. DataFrame.style requires jinja2. Use pip or conda to install Jinja2.
section time 280s
